# Benchmark — Feature Selection & Model Comparison

**Classification Transcriptomic with XAI**

This notebook runs the benchmark:
1. Load a gene expression dataset (CuMiDa format: `samples, type, probe_1, …`)
2. Apply each enabled feature selection method independently (Raw / mRMR / Boruta)
3. Benchmark all enabled models with Stratified K-Fold CV on each reduced dataset
4. **Run SHAP** on each trained model → plots + gene-level importance (probe→gene
   names from the dataset's local `annotation_file`)
5. Save per-fold metrics, summaries, trained models, and SHAP outputs to `outputs/`

Configure which datasets / methods / models run by editing the YAML files in `configs/`.

> Interpretation (association/tree rules) consumes the SHAP top genes produced here.

In [1]:
import sys
import warnings
from pathlib import Path

# Make src.* importable from the notebooks/ directory
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
warnings.filterwarnings("ignore", message=".*does not have many workers.*")

import numpy as np
import pandas as pd

from src.pipeline import GeneExpressionPipeline

## Configuration

Set **`RUN_MODE`** to pick how the benchmark runs:

| Mode | Datasets benchmarked |
|------|----------------------|
| `"single"`   | one dataset (`DATASET`) |
| `"multiple"` | many datasets (`DATASETS`, or `None` = all enabled in `datasets.yaml`) |

Which FS methods / models / SHAP top-k run is controlled by the YAML files in `configs/`.

In [16]:
# ── Run mode ──────────────────────────────────────────────────────────────
# "single"   → benchmark ONE dataset (DATASET).
# "multiple" → benchmark MANY datasets (DATASETS; None = all enabled).
RUN_MODE = "multiple"            # "single" | "multiple"

# ── Datasets ────────────────────────────────────────────────────────────────
DATASET  = "GEO-Breast-20711"
DATASETS = []   # e.g. ["GEO-Breast-20711", "GEO-Breast-20711-gene"] or None

# ── W&B (optional) ────────────────────────────────────────────────────────────
USE_WANDB = True

## 1. Initialise Pipeline

In [13]:
pipe = GeneExpressionPipeline(
    config_root='../configs',
    output_root='../outputs',
    use_wandb=USE_WANDB,
)

# Show enabled datasets / FS methods / models
pipe.print_summary()

  ⚠ W&B login failed: partially initialized module 'wandb' has no attribute 'data_types' (most likely due to a 
circular import). Continuing without W&B.

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Configuration Summary                                                                                           │
│ Classification Transcriptomic with XAI                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

    Project root : C:\Users\Bao Nguyen\LearningWithClaude\LV_code

    Config root  : C:\Users\Bao Nguyen\LearningWithClaude\LV_code\configs

                               Datasets (12 enabled)                                
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ status ┃                 dataset ┃                                   description ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      ✓ │        GEO-Breast-20711 │       Breast cancer – IHC subtypes (GSE20711) │
│      ✓ │  GEO-Mesothelioma-29354 │      Pleural mesothelioma subtypes (GSE29354) │
│      ✓ │        GEO-Kidney-36895 │    Clear-cell renal cell carcinoma (GSE36895) │
│      ✓ │         GEO-Colon-68468 │  Colon cancer molecular dissection (GSE68468) │
│      ✓ │   GEO-MultiCancer-68606 │ Multi-cancer interlaboratory study (GSE68606) │
│      ✓ │       CuMiDa-Ovary-6008 │                      Ovarian cancer (GSE6008) │
│      ✓ │      CuMiDa-Brain-15824 │               Brain tumor subtypes (GSE15824) │
│      ✓ │     CuMiDa-Breast-26304 │                      Breast cancer (GSE26304) │
│      ✓ │   CuMiDa-Leukemia-28497 │                  Leukemia subtypes (GSE28497) │
│      ✓ │ CuMiDa-Colorectal-41657 │                  Colorectal cancer (GSE41657) │
│      ✓ │     CuMiDa-Breast-45827 │             Breast cancer subtypes (GSE45827) │
│      ✓ │      CuMiDa-Brain-50161 │               Brain tumor subtypes (GSE50161) │
└────────┴─────────────────────────┴───────────────────────────────────────────────┘

    Models (7): nb, knn, svm, rf, dt, xgboost, ann

    Feature selection (4): raw, mrmr_k50, mrmr_k75, boruta

    Cross-validation : max 5-fold stratified

    W&B project      : transcriptomic_xai

## 2. Rare-class k sweep — full baseline + train/test-split baseline

Sweeps `feature_selection.yaml -> rare_class_k_values` for **both** baseline flows —

- **CV/full-data** (`pipe_full.run_benchmark(..., k_values=...)`) — writes to `outputs_baseline_full/k{k}/{dataset}/...`,
  raw + boruta + mrmr_k50/mrmr_k75, all enabled models.
- **Train/test-split** (`pipe.run_baseline_split(..., k_values=...)`) — writes to `outputs_baseline_split/k{k}/{dataset}/...`,
  boruta + mrmr_k50/mrmr_k75 only (no raw), all enabled models. This is also the **source of truth**
  `pipe.run_rule_extraction_holdout()` (notebook 04) reads its split + FS artifacts from for the same k.


In [ ]:
benchmark_datasets = [DATASET] if RUN_MODE == "single" else DATASETS
K_VALUES = pipe.config_loader.load_feature_selection_config().get("rare_class_k_values", [2,3, 4, 5, 6])
sweep_datasets = benchmark_datasets or pipe.config_loader.get_enabled_datasets()
print(f"Sweeping k={K_VALUES} for {len(sweep_datasets)} dataset(s) -> outputs_baseline_full/")

pipe_full = GeneExpressionPipeline(
    config_root='../configs',
    output_root='../outputs_baseline_full',
    use_wandb=USE_WANDB,
)
full_sweep_results = pipe_full.run_benchmark(sweep_datasets, k_values=K_VALUES)


In [ ]:
# Train/test-split baseline sweep -- also the split+FS source for notebook 04's rule extraction.
split_sweep_results = pipe.run_baseline_split(sweep_datasets, k_values=K_VALUES)


Interrupt due to electrictory --> rerun k=2 for baseline_full:  CuMiDa-Colorectal-41657, CuMiDa-Breast-45827,  CuMiDa-Brain-50161

In [14]:
sweep_datasets = [DATASET] if RUN_MODE == "single" else (DATASETS or None)

pipe_full = GeneExpressionPipeline(
    config_root='../configs',
    output_root='../outputs_baseline_full',
    use_wandb=True,
)
sweep_datasets = sweep_datasets or pipe_full.config_loader.get_enabled_datasets()
K_VALUES = pipe_full.config_loader.load_feature_selection_config().get("rare_class_k_values", [2, 3, 4, 5, 6])

# 1) Rerun k=2
k2_pending = ["CuMiDa-Colorectal-41657", "CuMiDa-Brain-50161", "CuMiDa-Breast-45827"]
pipe_full.run_benchmark(k2_pending, k_values=[2])

# 2) Continue with >=k3
remaining_k_values = [k for k in K_VALUES if k >= 3]
pipe_full.run_benchmark(sweep_datasets, k_values=remaining_k_values)

  ⚠ W&B login failed: partially initialized module 'wandb' has no attribute 'data_types' (most likely due to a 
circular import). Continuing without W&B.

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Benchmark — Feature Selection & Model Benchmarking                                                              │
│ Classification Transcriptomic with XAI                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

    Run mode: 3 datasets → CuMiDa-Colorectal-41657, CuMiDa-Brain-50161, CuMiDa-Breast-45827

──────────────────────────────────────────────── Rare-class k = 2 ─────────────────────────────────────────────────

───────────────────────────────────── Dataset: CuMiDa-Colorectal-41657 (k=2) ──────────────────────────────────────

...

  ✓ knn complete

  → Training SVM on raw

    [SVM] 5-fold CV · 86 samples × 33467 features

      fold 1/5 · acc=0.8889 · f1_macro=0.8995 · bal_acc=0.9000

KeyboardInterrupt: 